# 模型对比分析报告

本 Notebook 汇总三个 CNN 模型（MiniCNN / VGGLite / MicroResNet）的对比结果。

- **数据来源为显式指定**（`COMPARISON_SOURCES`），不按 mtime 自动挑选；缺文件/来源混合时直接失败。
- 训练历史与测试指标必须来自同一批显式来源；本 Notebook 的所有图表均**现场生成**
  （不再读取历史公共图片），评估产物保存到 `analysis/evaluations/`（含权重 SHA-256，可复算）。
- 当前来源：三个 legacy 导出权重 + legacy 训练日志（2026-05 历史产物，信息不全，已标注）；
  正式比较时替换为同一批 runs 的 checkpoint/history——**禁止 legacy/smoke/正式结果混用**。


In [ ]:
import sys

sys.path.insert(0, '..')

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

from training.trainer import PROJECT_ROOT

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

config_path = PROJECT_ROOT / 'configs' / 'training_config.yaml'
with open(config_path, encoding='utf-8') as f:
    config = yaml.safe_load(f)

CLASS_NAMES = config['data']['class_names']
MODEL_NAMES = ['mini_cnn', 'vgg_lite', 'micro_resnet']

# ------------------------------------------------------------------
# 数据来源：显式指定，不按 mtime 自动挑选（R09）
# 当前：三个 legacy 导出权重 + legacy 训练日志（2026-05 历史产物；来源与
#       SHA-256 见 inference/saved_models/export_manifest.json）。
# 正式比较时：把三组替换为同一批 runs 的 checkpoint/history（同一协议）；
#            禁止 legacy / smoke / 正式结果混用。
# ------------------------------------------------------------------
COMPARISON_SOURCES = {
    'mini_cnn': {
        'checkpoint': 'inference/saved_models/mini_cnn_best.pth',
        'history': 'training/logs/mini_cnn/history.json',
        'label': 'legacy（2026-05 历史产物）',
    },
    'vgg_lite': {
        'checkpoint': 'inference/saved_models/vgg_lite_best.pth',
        'history': 'training/logs/vgg_lite/history.json',
        'label': 'legacy（2026-05 历史产物）',
    },
    'micro_resnet': {
        'checkpoint': 'inference/saved_models/micro_resnet_best.pth',
        'history': 'training/logs/micro_resnet/history.json',
        'label': 'legacy（2026-05 历史产物）',
    },
}

# ---- 来源一致性校验（S04）：checkpoint 与 history 的绑定必须在生成任何图表前通过 ----
# formal：新格式 run（run_id/协议绑定；要求同一 run 的 history 且整组共享同一协议）
# legacy-unverified：旧产物缺元数据——仅作历史展示并标注，不与正式结果混排
from utils.comparison_check import validate_comparison_set

_source_reports = validate_comparison_set(
    {name: {"checkpoint": PROJECT_ROOT / src['checkpoint'],
            "history": PROJECT_ROOT / src['history']}
     for name, src in COMPARISON_SOURCES.items()}
)
SOURCE_VERIFIED = all(r["verdict"] == "formal" for r in _source_reports)
SOURCE_TAG = "" if SOURCE_VERIFIED else "【来源未验证】"

print("来源校验结果:")
for r in _source_reports:
    extra = f" | run={r['run_id']}" if r.get('run_id') else ""
    print(f"  {r['model_name']}: {r['verdict']} | sha256={r['checkpoint_sha256'][:16]}...{extra}")
if not SOURCE_VERIFIED:
    print()
    print("⚠️ " + "=" * 64)
    print("⚠️  本组来源为 legacy-unverified（缺 run/协议元数据，无法证明权重与日志同源）：")
    print("⚠️  以下所有图表均为【来源未验证】的历史展示，不得作为正式比较结论。")
    print("⚠️  正式比较请替换 COMPARISON_SOURCES 为同一冻结协议下的 runs。")
    print("⚠️ " + "=" * 64)

histories, history_sources = {}, {}
for name, src in COMPARISON_SOURCES.items():
    with open(PROJECT_ROOT / src['history'], encoding='utf-8') as f:
        histories[name] = json.load(f)
    history_sources[name] = src['label']
    n_epochs = len(histories[name]['train_loss'])
    print(f"已加载 {name} 训练历史 ({n_epochs} epochs) | 来源: {src['label']}")


## 1. 训练曲线对比（来源见上方加载输出）


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {'mini_cnn': 'blue', 'vgg_lite': 'green', 'micro_resnet': 'red'}

for name, hist in histories.items():
    epochs = range(1, len(hist['train_loss']) + 1)
    c = colors[name]
    axes[0, 0].plot(epochs, hist['train_loss'], color=c, label=f'{name}', linestyle='-')
    axes[0, 1].plot(epochs, hist['val_loss'], color=c, label=f'{name}', linestyle='-')
    axes[1, 0].plot(epochs, hist['train_acc'], color=c, label=f'{name}', linestyle='-')
    axes[1, 1].plot(epochs, hist['val_acc'], color=c, label=f'{name}', linestyle='-')

axes[0, 0].set_title('Train Loss')
axes[0, 0].legend()
axes[0, 1].set_title('Val Loss')
axes[0, 1].legend()
axes[1, 0].set_title('Train Accuracy')
axes[1, 0].legend()
axes[1, 1].set_title('Val Accuracy')
axes[1, 1].legend()

for ax in axes.flat:
    ax.set_xlabel('Epoch')

plt.suptitle(f'{SOURCE_TAG}三模型训练曲线对比（来源见上方校验输出）', fontsize=12)
plt.tight_layout()
curve_dir = PROJECT_ROOT / 'analysis' / 'training_curves'
curve_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(curve_dir / 'comparison_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. 训练历史汇总（Val 口径；来源见加载输出）


In [ ]:
# 汇总表：参数量来自各 checkpoint 的实际 model_spec（S04：不从当前 YAML 构建）；
# Val 指标来自各自训练历史（来源见最后两列）
from inference.infer_utils import describe_checkpoint

summary = []
for name in MODEL_NAMES:
    if name not in histories:
        continue
    hist = histories[name]
    best_epoch = int(np.argmax(hist['val_acc'])) + 1
    ckpt_info = describe_checkpoint(PROJECT_ROOT / COMPARISON_SOURCES[name]['checkpoint'])
    summary.append({
        '模型': name,
        '参数量': f"{ckpt_info['params_total']:,}",
        '权重SHA256(前16)': ckpt_info['sha256'][:16],
        '最优Epoch': best_epoch,
        '最优Val Acc': f"{hist['val_acc'][best_epoch - 1]:.4f}",
        '最优Val Loss': f"{hist['val_loss'][best_epoch - 1]:.4f}",
        '最终Train Acc': f"{hist['train_acc'][-1]:.4f}",
        '总Epochs': len(hist['train_loss']),
        '来源状态': next(r['verdict'] for r in _source_reports if r['model_name'] == name),
        '历史来源': history_sources[name],
    })

summary_df = pd.DataFrame(summary)
print("=== 模型性能汇总（Val=验证集口径，与测试指标分开列示）===")
summary_df


## 3. 统一评估（显式 checkpoint，结果保存到 analysis/evaluations/）


In [ ]:
# 统一评估入口：对显式来源的 checkpoint 评估 PrivateTest（可追溯到权重 SHA-256）
# 评估产物（指标 + 逐样本预测 + 概率）保存到 analysis/evaluations/comparison_*/
from utils.evaluation import evaluate_checkpoint

SPLIT = 'PrivateTest'

eval_results = {}
for name, src in COMPARISON_SOURCES.items():
    ckpt_path = PROJECT_ROOT / src['checkpoint']
    print(f'--- {name} ---')
    eval_results[name] = evaluate_checkpoint(
        ckpt_path, split=SPLIT, device='cpu',
        output_dir=PROJECT_ROOT / 'analysis' / 'evaluations' / f'comparison_{name}_{SPLIT}',
    )

verdict_by_name = {r['model_name']: r['verdict'] for r in _source_reports}
rows = []
for name, r in eval_results.items():
    m = r['metrics']
    rows.append({
        '模型': name,
        'accuracy': f"{m['accuracy']:.4f}",
        'macro_f1': f"{m['macro_f1']:.4f}",
        'balanced_acc': f"{m['balanced_accuracy']:.4f}",
        'ECE(15bin)': f"{m['ece']:.4f}",
        '权重SHA256(前16)': r['checkpoint']['sha256'][:16],
        '来源状态': verdict_by_name.get(name, '?'),
    })
print(f"\n=== {SPLIT} 指标汇总（统一评估入口）===")
display(pd.DataFrame(rows))


## 4. 混淆矩阵（现场生成，不读取历史图片）


In [ ]:
# 混淆矩阵（现场从统一评估的逐样本预测生成；不读取历史公共图片）
if 'eval_results' not in globals() or not eval_results:
    raise RuntimeError('请先运行上方「统一评估」单元')

fig, axes = plt.subplots(1, len(eval_results), figsize=(7 * len(eval_results), 6))
if len(eval_results) == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, eval_results.items(), strict=True):
    cm = np.array(r['metrics']['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, cbar=False)
    ax.set_title(f"{SOURCE_TAG}{name}（acc={r['metrics']['accuracy']:.4f}）")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle(f'{SOURCE_TAG}混淆矩阵（现场生成，{SPLIT}，统一评估入口）', fontsize=13)
plt.tight_layout()
out = PROJECT_ROOT / 'analysis' / 'comparison_confusion_matrices.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()


## 5. 各类别 Recall 与参数量-准确率（现场生成）


In [ ]:
# 各类别 Recall 对比 + 参数量-准确率散点（均现场生成）
from utils.model_spec import build_model_from_spec, count_parameters, make_spec_from_config

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(CLASS_NAMES))
width = 0.25
colors = {'mini_cnn': '#3498db', 'vgg_lite': '#2ecc71', 'micro_resnet': '#e74c3c'}
for i, (name, r) in enumerate(eval_results.items()):
    values = [r['metrics']['per_class'][cls]['recall'] for cls in CLASS_NAMES]
    bars = ax1.bar(x + i * width, values, width, label=name,
                   color=colors.get(name), alpha=0.85)
    for bar, val in zip(bars, values, strict=True):
        ax1.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax1.set_xlabel('情感类别')
ax1.set_ylabel('Recall')
ax1.set_title(f'{SOURCE_TAG}各类别 Recall 对比（{SPLIT}）')
ax1.set_xticks(x + width)
ax1.set_xticklabels(CLASS_NAMES)
ax1.set_ylim(0, 1.1)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

for name in eval_results:
    spec = make_spec_from_config(config, name)
    total, _ = count_parameters(build_model_from_spec(spec, allow_unknown_dropout=True))
    acc = eval_results[name]['metrics']['accuracy']
    ax2.scatter(total, acc, s=200, zorder=5)
    ax2.annotate(name, (total, acc), textcoords='offset points', xytext=(10, 10), fontsize=12)
ax2.set_xlabel('参数量')
ax2.set_ylabel(f'{SPLIT} accuracy')
ax2.set_title(f'{SOURCE_TAG}参数量 vs 准确率')
ax2.set_xscale('log')

plt.tight_layout()
out = PROJECT_ROOT / 'analysis' / 'comparison_param_efficiency.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()

first = eval_results[next(iter(eval_results))]
support = first['metrics']['per_class']['Disgust']['support']
print(f"注: {SPLIT} Disgust 支持数为 {support}；"
      f"一张样本约对应 {100 / support:.2f} 个百分点 recall 变化，解读时注意粒度")


## 6. 结论与模型选择

- 准确率最优与参数效率最优是否一致（见 §3 汇总表与 §5 散点图）
- 各模型类别差异（见 §5 Recall 对比；同时参考 Macro-F1 与 balanced accuracy）
- 部署：使用 `tools/export_model.py` 显式导出所选 run 的 best.pth；
  本 Notebook 不自动复制任何文件


In [ ]:
# 结论汇总：不自动复制文件；导出请使用 tools/export_model.py
if eval_results:
    best_name = max(eval_results, key=lambda n: eval_results[n]['metrics']['accuracy'])
    m = eval_results[best_name]['metrics']
    print(f"评估口径: {SPLIT}（统一评估入口；来源=COMPARISON_SOURCES 显式指定）")
    src_state = 'formal' if SOURCE_VERIFIED else 'legacy-unverified（不可作为正式比较结论）'
    print(f"来源状态: {src_state}")
    print(f"  accuracy 最优: {best_name} ({m['accuracy']:.4f})")
    print("注意: accuracy 与 Macro-F1 排名可能不一致，请联合讨论；")
    print("      本 Notebook 的图与表均来自同一批显式来源的 checkpoint 与评估结果")
    print()
    print("导出命令示例:")
    print("  python tools/export_model.py --list  # 查看已有导出与来源")
    print(f"  python tools/export_model.py --checkpoint "
          f"training/runs/{best_name}/<run_id>/checkpoints/best.pth")
else:
    print("无可用评估数据")
